# 05 — Agent de tarification dynamique (Dev C)

Entraînement du réseau dense et comparaison avec un prix fixe dans le simulateur.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

In [ ]:
from src.simulation.run_simulation import generate_synthetic_data

# Remplacer par src.data.load_data.* une fois les données brutes disponibles
consumption, solar = generate_synthetic_data(n_days=120)

In [ ]:
from src.models.pricing_dense import PricingAgent

hours = consumption['timestamp'].dt.hour.to_numpy()
rng = np.random.default_rng(0)
states = np.column_stack([consumption['consumption_kwh'], solar['pv_power'], rng.uniform(0, 1, len(hours)),
                          np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24)])
agent = PricingAgent()
history = agent.fit(states)

In [ ]:
from src.simulation.microgrid_env import fixed_price_policy
from src.simulation.run_simulation import simulate, summarize
from src.evaluation.plots import plot_simulation

demand, pv = consumption['consumption_kwh'].to_numpy(), solar['pv_power'].to_numpy()
results = {name: simulate(demand, pv, pol, hours) for name, pol in {'prix_fixe': fixed_price_policy(), 'agent': agent.act}.items()}
pd.DataFrame({k: summarize(v) for k, v in results.items()})

In [ ]:
plot_simulation(results['agent']);